In [0]:
from pyspark.sql.functions import *

In [0]:
df=spark.table('shop_stream.core.orders')
df.display()

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code
L0000001,O000001,C00628,P0123,1,106.28,2026-02-20T07:14:43.000Z,completed,FESTIVE20
L0000002,O000001,C00628,P0096,1,267.14,2026-02-20T07:14:43.000Z,completed,WELCOME15
L0000003,O000002,C00141,P0057,2,22.09,2026-02-07T07:02:26.000Z,completed,null
L0000004,O000002,C00141,P0017,1,53.8,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000005,O000002,C00141,P0126,1,107.67,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000006,O000002,C00141,P0110,1,27.62,2026-02-07T07:02:26.000Z,completed,null
L0000007,O000003,C00845,P0165,2,63.33,2026-04-24T10:25:39.000Z,cancelled,null
L0000008,O000004,C00463,P0011,1,64.02,2026-04-27T01:56:15.000Z,returned,null
L0000009,O000004,C00463,P0113,1,319.88,2026-04-27T01:56:15.000Z,returned,null
L0000010,O000004,C00463,P0074,1,156.41,2026-04-27T01:56:15.000Z,returned,WELCOME15


In [0]:
df.createOrReplaceTempView("orders")

In [0]:
result=spark.sql("""
    select status,count(*) from orders
    group by status
    order by count(*) desc
          """)
display(result)

status,count(*)
completed,8480
cancelled,2071
returned,2053
COMPLETED,731
RETURNED,201
CANCELLED,181


In [0]:
df = df.withColumn("status", initcap(col("status")))

In [0]:
df.createOrReplaceTempView("orders")
result=spark.sql("""
    select status,count(*) from orders
    group by status
    order by count(*) desc
          """)
display(result)

status,count(*)
Completed,9211
Returned,2254
Cancelled,2252


In [0]:
# Check for duplicates based on order_line_id
df.groupBy("order_line_id").count().filter(col("count") > 1).display()

order_line_id,count
L0000040,2
L0000059,2
L0000108,2
L0000122,2
L0000170,2
L0000240,2
L0000246,2
L0000391,2
L0000421,2
L0000445,2


In [0]:
# Drop duplicates based on order_line_id
df = df.dropDuplicates(["order_line_id"])

In [0]:
df.groupBy("order_line_id").count().filter(col("count") > 1).display()

order_line_id,count


In [0]:
df.filter(col("quantity") < 0).display()

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code
L0000028,O000014,C00748,P0148,-1,153.07,2026-06-17T02:30:46.000Z,Cancelled,SAVE10
L0000030,O000016,C00002,P0158,-1,317.86,2026-01-01T14:40:50.000Z,Completed,WELCOME15
L0000058,O000037,C00814,P0157,-1,310.93,2026-06-10T13:45:58.000Z,Completed,WELCOME15
L0000077,O000046,C00379,P0088,-1,162.05,2026-02-09T22:15:31.000Z,Completed,null
L0000246,O000138,C00731,P0091,-1,107.19,2026-04-23T13:47:38.000Z,Completed,null
L0000280,O000157,C00009,P0142,-1,33.62,2026-01-21T22:27:26.000Z,Completed,null
L0000336,O000188,C00833,P0098,-1,148.36,2026-04-27T19:18:41.000Z,Completed,null
L0000346,O000194,C00506,P0051,-1,83.38,2026-03-02T08:31:57.000Z,Cancelled,WELCOME15
L0000391,O000217,C00692,P0155,-1,209.56,2026-03-09T09:47:52.000Z,Cancelled,null
L0000410,O000228,C00969,P0106,-1,139.53,2026-01-08T05:17:56.000Z,Cancelled,null


In [0]:
df=df.filter(col("quantity") > 0)

In [0]:
df.count()

13228

In [0]:
df.write.mode("overwrite").format('delta').saveAsTable("shop_stream.silver.orders")